<a href="https://colab.research.google.com/github/Sheulynath/Assignment/blob/Request/news2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip uninstall -y onnx onnx-tf tensorflow tensorflow-addons tensorflow-probability
!pip install tensorflow==2.15.0 tensorflow-addons==0.22.0 tensorflow-probability==0.22.0
!pip install onnx==1.14.1 onnx-tf==1.10.0


Found existing installation: onnx 1.14.1
Uninstalling onnx-1.14.1:
  Successfully uninstalled onnx-1.14.1
Found existing installation: onnx-tf 1.10.0
Uninstalling onnx-tf-1.10.0:
  Successfully uninstalled onnx-tf-1.10.0
Found existing installation: tensorflow 2.15.0
Uninstalling tensorflow-2.15.0:
  Successfully uninstalled tensorflow-2.15.0
Found existing installation: tensorflow-addons 0.22.0
Uninstalling tensorflow-addons-0.22.0:
  Successfully uninstalled tensorflow-addons-0.22.0
  Using cached tensorflow-2.15.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.4 kB)
  Using cached tensorflow_addons-0.22.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.8 kB)
Using cached tensorflow-2.15.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (475.3 MB)
Using cached tensorflow_addons-0.22.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (612 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 18.5 MB/s eta 0:00:00

In [ ]:
import onnx
from onnx_tf.backend import prepare

# Load ONNX model
onnx_model = onnx.load('/content/graphad_model.onnx')

# Convert to TensorFlow
tf_rep = prepare(onnx_model)

# Export to TensorFlow SavedModel format
tf_rep.export_graph('tf_model')


/usr/local/lib/python3.11/dist-packages/tensorflow_addons/utils/tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(
/usr/local/lib/python3.11/dist-packages/tensorflow_addons/utils/ensure_tf_install.py:53: UserWarning: Tensorflow Addons supports using Python ops for all Tensorflow versions above or equal to 2.12.0 and strictly below 2.15.0 (nightly versions are not supported). 
 The versions of TensorFlow you are currently using is 2.15.0 and is not supported. 
Some things might work, some things might not.
If you were to encounter a bug, do not file an issue.
If you want to make sure you're u

In [ ]:
import shutil

# Zip the folder
shutil.make_archive("tf_model", 'zip', "tf_model")


'/content/tf_model.zip'

In [ ]:
from google.colab import files

# Download the zipped model
files.download("tf_model.zip")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, BatchNormalization, Dropout
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, classification_report
converter = tf.lite.TFLiteConverter.from_keras_model(onnx_model)
onnx_model = converter.convert()

with open("model.tflite", 'wb') as f:
  f.write(onnx_model)

AttributeError: call

In [ ]:
!pip install onnxruntime

import onnxruntime as ort
import numpy as np

# Load ONNX model
onnx_session = ort.InferenceSession("graphad_model.onnx")
input_name = onnx_session.get_inputs()[0].name

# Sample input (make sure it's the correct shape and dtype)
input_data = np.random.rand(1, 3, 224, 224).astype(np.float32)

# Get ONNX output
onnx_output = onnx_session.run(None, {input_name: input_data})


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.5 MB/s eta 0:00:00


In [ ]:
import tensorflow as tf

# Load TF model
tf_model = tf.saved_model.load("tf_model")

# Run inference
infer = tf_model.signatures["serving_default"]
# You may need to adapt input key name (print infer.structured_input_signature)
tf_output = infer(tf.constant(input_data))  # replace key if needed


In [ ]:
# Get TensorFlow model output tensor
tf_output_array = list(tf_output.values())[0].numpy()

# Compare both outputs
is_equivalent = np.allclose(onnx_output[0], tf_output_array, atol=1e-4)

print("Outputs are equivalent:", is_equivalent)


Outputs are equivalent: True


In [ ]:
import tensorflow as tf

# Load your Keras model or TensorFlow SavedModel
# For Keras model:
# model = tf.keras.models.load_model('model.h5')

# OR for SavedModel directory:
saved_model_dir = 'tf_model'

converter = tf.lite.TFLiteConverter.from_saved_model(saved_model_dir)
tflite_model = converter.convert()

# Save the TensorFlow Lite model to a single .tflite file
with open('model.tflite', 'wb') as f:
    f.write(tflite_model)
